# 🔌 Projeto: Engenharia de Dados em Big Data
## Onde Construir um Datacenter no Brasil?
### Etapa 2 — Análise Exploratória e Limpeza de Dados (Silver → Gold)

**Disciplina:** Fundamentos de Dados e Analytics — Engenharia de Dados em Big Data  
**Profs.:** Fabio Rossi Versolatto · Gustavo Moreira Calixto

> **Fonte de dados:** 100% MongoDB Atlas (camada Bronze já ingerida na Etapa 1).  
> Esta etapa transforma Bronze → Silver → Gold e realiza a EDA completa.

---

## 0.1 — Instalação de dependências

In [ ]:
!pip install -q pymongo dnspython plotly xgboost openpyxl

## 0.2 — Imports

In [ ]:
import os, re
import numpy as np
import pandas as pd
from datetime import datetime
from pathlib import Path
from scipy.spatial import cKDTree
from pymongo import MongoClient, ASCENDING
from pymongo.errors import BulkWriteError
import warnings
warnings.filterwarnings('ignore')
print('✅ Imports prontos.')

## 0.3 — Conexão MongoDB Atlas

> **Pré-requisito:** secret `MONGO_URI` cadastrado em *Colab → 🔑 Secrets*  
> (mesmo procedimento da Etapa 1)

In [ ]:
from google.colab import userdata

MONGO_URI = userdata.get('MONGO_URI')
assert MONGO_URI, '❌ Cadastre o secret MONGO_URI em Configurações → Secrets'

client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=15000)
client.admin.command('ping')
db = client['aneel_datacenters_v2']
print('✅ Conectado ao MongoDB Atlas. Banco:', db.name)
print('📋 Coleções disponíveis:', sorted(db.list_collection_names()))

## 0.4 — Funções utilitárias

In [ ]:
def colecao_tem_dados(nome, minimo=1):
    """Retorna True se a coleção tem >= `minimo` documentos."""
    try:
        return db[nome].estimated_document_count() >= minimo
    except Exception:
        return False

def verificar_bronze():
    """Verifica que todas as coleções Bronze estão populadas antes de prosseguir."""
    bronze_esperadas = [
        'bronze_siga',
        'bronze_inmet_estacoes',
        'bronze_anatel_compartilhamento',
        'bronze_anatel_credenciadas',
        'bronze_anatel_interconexao',
        'bronze_anatel_mvno',
        'bronze_anatel_ran_sharing',
        'bronze_datacenters',
    ]
    ok = True
    for nome in bronze_esperadas:
        n = db[nome].estimated_document_count()
        status = '✅' if n else '❌'
        print(f'{status} {nome:40s} {n:>10,} docs')
        if not n:
            ok = False
    if not ok:
        raise RuntimeError(
            '❌ Uma ou mais coleções Bronze estão vazias.\n'
            'Execute a Etapa 1 (ingestão) antes de prosseguir.'
        )
    print('\n✅ Todas as coleções Bronze verificadas com sucesso.')

verificar_bronze()

---
# 🟨 ETAPA 2 — Análise Exploratória e Limpeza (Silver + Gold)

> A camada **Silver** limpa, tipa e valida os dados Bronze.  
> A camada **Gold** agrega features prontas para ML.

## 2.0 — Atalho: recarregar Silver/Gold do Mongo se já existirem

In [ ]:
SKIP_RECOMPUTE = all(
    colecao_tem_dados(c) for c in
    ['silver_siga', 'silver_inmet', 'silver_anatel_uf',
     'silver_datacenters', 'gold_empreendimentos_features']
)

if SKIP_RECOMPUTE:
    print('⚡ Carregando Silver/Gold do MongoDB (sem recomputar)...')
    df_siga       = pd.DataFrame(list(db['silver_siga'].find({}, {'_id': 0})))
    silver_inmet  = pd.DataFrame(list(db['silver_inmet'].find({}, {'_id': 0})))
    silver_anatel = pd.DataFrame(list(db['silver_anatel_uf'].find({}, {'_id': 0})))
    silver_dc     = pd.DataFrame(list(db['silver_datacenters'].find({}, {'_id': 0})))
    emp           = pd.DataFrame(list(db['gold_empreendimentos_features'].find({}, {'_id': 0})))
    print(f'   silver_siga:                   {df_siga.shape}')
    print(f'   silver_inmet:                  {silver_inmet.shape}')
    print(f'   silver_anatel_uf:              {silver_anatel.shape}')
    print(f'   silver_datacenters:            {silver_dc.shape}')
    print(f'   gold_empreendimentos_features: {emp.shape}')
else:
    print('🔁 Silver/Gold ausentes — será recomputado nas células seguintes.')

## 2.1 — Silver: limpeza e tipagem do SIGA-ANEEL

Origem: `bronze_siga` (MongoDB) → `silver_siga`

Operações:
- Campos numéricos em formato BR (`1.234,56`) → float
- Datas → datetime
- Coordenadas (graus/minutos/segundos ou decimal) → float WGS-84
- Sanitização de coordenadas fora do Brasil
- Flag `eh_renovavel`

In [ ]:
if not SKIP_RECOMPUTE:
    def to_float_br(s):
        """Converte string numérica BR ('1.234,56') -> float."""
        if pd.isna(s) or s in ('', '-', 'N/A', 'nan'):
            return np.nan
        s = str(s).replace('.', '').replace(',', '.')
        try:
            return float(s)
        except ValueError:
            return np.nan

    def parse_coord(s):
        """Converte coordenada texto (decimal ou GMS) → float."""
        if pd.isna(s) or str(s).strip() == '':
            return np.nan
        s = str(s).strip()
        try:
            return float(s.replace(',', '.'))
        except Exception:
            pass
        m = re.match(r"(\d+)[°\s]+(\d+)['′\s]+([\d\.,]+)?\"?\s*([NSEWO])", s)
        if m:
            d, mi, se, hem = m.groups()
            se = float((se or '0').replace(',', '.'))
            val = float(d) + float(mi) / 60 + se / 3600
            if hem in ('S', 'W', 'O'):
                val = -val
            return val
        return np.nan

    # ── Carrega Bronze SIGA do MongoDB ───────────────────────────────────────
    df_siga = pd.DataFrame(list(db['bronze_siga'].find(
        {}, {'_id': 0, '_ingest_ts': 0, '_source': 0, '_source_url': 0}
    )))
    print('Bronze SIGA shape:', df_siga.shape)

    # Numéricos
    for c in ['MdaPotenciaOutorgadaKw', 'MdaPotenciaFiscalizadaKw', 'MdaGarantiaFisicaKw']:
        if c in df_siga.columns:
            df_siga[c] = df_siga[c].apply(to_float_br)

    # Datas
    for c in ['DatGeracaoConjuntoDados', 'DatEntradaOperacao',
              'DatInicioVigencia', 'DatFimVigencia']:
        if c in df_siga.columns:
            df_siga[c] = pd.to_datetime(df_siga[c], errors='coerce')

    # Coordenadas
    df_siga['lat'] = df_siga.get(
        'NumCoordNEmpreendimento', pd.Series([np.nan] * len(df_siga))
    ).apply(parse_coord)
    df_siga['lon'] = df_siga.get(
        'NumCoordEEmpreendimento', pd.Series([np.nan] * len(df_siga))
    ).apply(parse_coord)

    # Sanitiza coords fora do Brasil
    df_siga.loc[~df_siga['lat'].between(-35, 6),   'lat'] = np.nan
    df_siga.loc[~df_siga['lon'].between(-75, -30), 'lon'] = np.nan

    # Flag renovável
    RENOVAVEL = ['HÍDRICA', 'EÓLICA', 'RADIAÇÃO SOLAR', 'BIOMASSA', 'MARÉS', 'HIDRAULICA']
    df_siga['eh_renovavel'] = df_siga.get('DscOrigemCombustivel', pd.Series('')).isin(RENOVAVEL)

    print(f'✅ Silver SIGA. Com lat/lon: '
          f'{df_siga[["lat","lon"]].notna().all(axis=1).sum():,} empreendimentos')

    # Persiste Silver SIGA
    col_silver_siga = db['silver_siga']
    col_silver_siga.drop()
    silver_docs = df_siga.copy()
    for c in ['DatGeracaoConjuntoDados', 'DatEntradaOperacao',
              'DatInicioVigencia', 'DatFimVigencia']:
        if c in silver_docs.columns:
            silver_docs[c] = silver_docs[c].apply(
                lambda x: x.isoformat() if pd.notna(x) else None
            )
    col_silver_siga.insert_many(silver_docs.to_dict(orient='records'))
    col_silver_siga.create_index([('SigUFPrincipal', 1), ('DscFaseUsina', 1)])
    print(f'✅ silver_siga persistida: {col_silver_siga.estimated_document_count():,} docs')

df_siga[['NomEmpreendimento', 'SigUFPrincipal', 'SigTipoGeracao', 'DscFaseUsina',
         'MdaPotenciaOutorgadaKw', 'MdaPotenciaFiscalizadaKw',
         'lat', 'lon', 'eh_renovavel']].head()

## 2.2 — Silver: agregação climática INMET por estação

Origem: `bronze_inmet_estacoes` (MongoDB) → `silver_inmet`

Variáveis climáticas agregadas por estação (média, std, min, max do trimestre):

| Variável                 | Por que importa para um DC           |
|--------------------------|--------------------------------------|
| Temperatura do ar (°C)   | Custo de refrigeração — PUE          |
| Umidade relativa (%)     | Corrosão e condensação               |
| Precipitação (mm)        | Risco de enchente                    |
| Radiação global (kJ/m²)  | Potencial fotovoltaico               |
| Vento — rajada (m/s)     | Risco estrutural                     |

In [ ]:
if not SKIP_RECOMPUTE:
    def converter_num(x):
        if pd.isna(x) or str(x).strip() in ('', '-9999', 'null'):
            return np.nan
        try:
            return float(str(x).replace(',', '.'))
        except Exception:
            return np.nan

    COLS_NUM = {
        'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)':              'prec_mm',
        'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)': 'temp_c',
        'UMIDADE RELATIVA DO AR, HORARIA (%)':           'umid_rel',
        'RADIACAO GLOBAL (Kj/m²)':                       'radiacao',
        'VENTO, RAJADA MAXIMA (m/s)':                    'vento_rajada',
        'VENTO, VELOCIDADE HORARIA (m/s)':               'vento_vel',
    }

    # ── Carrega observações horárias do MongoDB ───────────────────────────────
    print('⏳ Carregando bronze_inmet_obs do MongoDB...')
    df_obs_full = pd.DataFrame(list(db['bronze_inmet_obs'].find({}, {'_id': 0})))
    print('Shape obs INMET:', df_obs_full.shape)

    ren = {k: v for k, v in COLS_NUM.items() if k in df_obs_full.columns}
    df_obs = df_obs_full.rename(columns=ren).copy()
    for v in ren.values():
        df_obs[v] = df_obs[v].apply(converter_num)

    print('Colunas climáticas disponíveis:', list(ren.values()))

    # ── Metadados das estações ────────────────────────────────────────────────
    estacoes_df = pd.DataFrame(list(db['bronze_inmet_estacoes'].find(
        {}, {'_id': 0, '_ingest_ts': 0, '_source': 0}
    )))

    # Agrega por estação
    agg = (df_obs.groupby('ESTACAO_CODIGO')
           .agg(
               temp_media   =('temp_c',       'mean'),
               temp_std     =('temp_c',       'std'),
               temp_max     =('temp_c',       'max'),
               temp_min     =('temp_c',       'min'),
               umid_media   =('umid_rel',     'mean'),
               prec_total   =('prec_mm',      'sum'),
               prec_max_h   =('prec_mm',      'max'),
               radiacao_med =('radiacao',     'mean'),
               vento_med    =('vento_vel',    'mean'),
               rajada_max   =('vento_rajada', 'max'),
               n_obs        =('temp_c',       'count'),
           )
           .reset_index())

    silver_inmet = estacoes_df.merge(
        agg, left_on='CODIGO (WMO)', right_on='ESTACAO_CODIGO', how='left'
    )
    silver_inmet = silver_inmet.dropna(subset=['LATITUDE', 'LONGITUDE'])
    print(f'✅ Silver INMET: {len(silver_inmet)} estações com clima agregado')

    # Persiste Silver INMET
    col_silver_inmet = db['silver_inmet']
    col_silver_inmet.drop()
    col_silver_inmet.insert_many(silver_inmet.to_dict(orient='records'))
    col_silver_inmet.create_index([('UF', 1)])
    print(f'✅ silver_inmet persistida: {col_silver_inmet.estimated_document_count()} docs')

silver_inmet[['REGIAO', 'UF', 'ESTACAO', 'CODIGO (WMO)', 'LATITUDE', 'LONGITUDE',
              'temp_media', 'umid_media', 'prec_total', 'radiacao_med', 'vento_med']].head()

## 2.3 — Silver: agregação ANATEL por UF

Origem: 5 coleções `bronze_anatel_*` (MongoDB) → `silver_anatel_uf`

Proxy de cobertura de telecom/fibra por UF: contagem de contratos e menções nos textos.

In [ ]:
if not SKIP_RECOMPUTE:
    UF_LIST = [
        'AC','AL','AM','AP','BA','CE','DF','ES','GO','MA','MG','MS','MT',
        'PA','PB','PE','PI','PR','RJ','RN','RO','RR','RS','SC','SE','SP','TO'
    ]

    def contar_uf(df, *cols_texto):
        """Conta menções a cada UF nas colunas de texto do DataFrame."""
        text = pd.Series('', index=df.index)
        for c in cols_texto:
            if c in df.columns:
                text = text.str.cat(df[c].fillna(''), sep=' ')
        return pd.Series({
            uf: text.str.contains(rf'\b{uf}\b', case=False, regex=True, na=False).sum()
            for uf in UF_LIST
        })

    COLECOES_ANATEL = [
        'compartilhamento', 'interconexao', 'mvno', 'ran_sharing', 'credenciadas'
    ]

    linhas_uf = []
    for nome in COLECOES_ANATEL:
        col_name = f'bronze_anatel_{nome}'
        df_a = pd.DataFrame(list(
            db[col_name].find({}, {'_id': 0, '_ingest_ts': 0, '_source': 0})
        ))
        if df_a.empty:
            print(f'⚠️  {col_name} vazia no MongoDB — pulando.')
            continue
        print(f'   • {nome:20s} → {df_a.shape}')
        cols_texto = [
            c for c in df_a.columns
            if any(k in c.upper() for k in
                   ('OBSERVACAO', 'PRESTADORA', 'DETENTORA', 'SOLICITANTE',
                    'CREDENCIADA', 'OBS'))
        ]
        linhas_uf.append(contar_uf(df_a, *cols_texto).rename(f'anatel_{nome}'))

    silver_anatel = (
        pd.concat(linhas_uf, axis=1)
        .reset_index()
        .rename(columns={'index': 'UF'})
    )
    silver_anatel['anatel_total'] = silver_anatel.iloc[:, 1:].sum(axis=1)
    silver_anatel = silver_anatel.sort_values('anatel_total', ascending=False)

    # Persiste Silver ANATEL
    col_silver_anatel = db['silver_anatel_uf']
    col_silver_anatel.drop()
    col_silver_anatel.insert_many(silver_anatel.to_dict(orient='records'))
    print(f'✅ silver_anatel_uf persistida: {col_silver_anatel.estimated_document_count()} UFs')

print('✅ Silver ANATEL por UF:')
display(silver_anatel.head(5))

## 2.4 — Silver: datacenters no Brasil (geocodificação)

Origem: `bronze_datacenters` (MongoDB) → `silver_datacenters`

Junta coordenadas aproximadas das principais cidades brasileiras.

In [ ]:
CIDADES_COORD = {
    'São Paulo':             (-23.5505, -46.6333),
    'Rio de Janeiro':        (-22.9068, -43.1729),
    'Belo Horizonte':        (-19.9167, -43.9345),
    'Brasília':              (-15.7939, -47.8828),
    'Curitiba':              (-25.4284, -49.2733),
    'Porto Alegre':          (-30.0346, -51.2177),
    'Fortaleza':             ( -3.7172, -38.5433),
    'Salvador':              (-12.9714, -38.5014),
    'Recife':                ( -8.0476, -34.8770),
    'Campinas':              (-22.9099, -47.0626),
    'Florianópolis':         (-27.5954, -48.5480),
    'Goiânia':               (-16.6869, -49.2648),
    'Manaus':                ( -3.1190, -60.0217),
    'Belém':                 ( -1.4558, -48.4902),
    'Natal':                 ( -5.7945, -35.2110),
    'João Pessoa':           ( -7.1153, -34.8610),
    'São José dos Campos':   (-23.2237, -45.9009),
    'São José do Rio Preto': (-20.8113, -49.3758),
    'Barueri':               (-23.5111, -46.8761),
    'Osasco':                (-23.5320, -46.7919),
    'Santana de Parnaíba':   (-23.4438, -46.9176),
    'Jundiaí':               (-23.1858, -46.8979),
    'Guarulhos':             (-23.4538, -46.5333),
    'Hortolândia':           (-22.8584, -47.2200),
    'Vitória':               (-20.3155, -40.3128),
    'Caxias do Sul':         (-29.1678, -51.1794),
    'Eusébio':               ( -3.8900, -38.4506),
    'Anápolis':              (-16.3267, -48.9528),
    'Paulínia':              (-22.7611, -47.1542),
}

if not SKIP_RECOMPUTE:
    # ── Carrega Bronze Datacenters do MongoDB ─────────────────────────────────
    silver_dc = pd.DataFrame(list(
        db['bronze_datacenters'].find({}, {'_id': 0, '_ingest_ts': 0, '_source': 0})
    ))
    silver_dc['LATITUDE']  = silver_dc['city'].map(
        lambda c: CIDADES_COORD.get(c, (None, None))[0]
    )
    silver_dc['LONGITUDE'] = silver_dc['city'].map(
        lambda c: CIDADES_COORD.get(c, (None, None))[1]
    )
    cobertura = silver_dc['LATITUDE'].notna().mean()
    print(f'✅ silver_datacenters: {silver_dc.shape}  cobertura geográfica: {cobertura:.0%}')

    # Persiste Silver Datacenters
    col_silver_dc = db['silver_datacenters']
    col_silver_dc.drop()
    col_silver_dc.insert_many(
        silver_dc.assign(_ingest_ts=datetime.utcnow(), _source='silver_datacenters')
                 .to_dict(orient='records')
    )
    col_silver_dc.create_index([('city', ASCENDING)])
    print(f'✅ silver_datacenters persistida: {col_silver_dc.estimated_document_count():,} docs')

silver_dc.head()

## 2.5 — Join geoespacial via KDTree

Para cada empreendimento de energia (com coordenadas):
1. Encontra a **estação INMET mais próxima** (features climáticas)
2. Conta **datacenters a ≤ 100 km** (saturação do mercado)
3. Associa **contratos ANATEL** da UF (cobertura de telecom)

In [ ]:
if not SKIP_RECOMPUTE:
    # ── 1) Estação INMET mais próxima ─────────────────────────────────────────
    inmet_xy   = silver_inmet[['LATITUDE', 'LONGITUDE']].to_numpy()
    tree_inmet = cKDTree(inmet_xy)

    emp = df_siga.dropna(subset=['lat', 'lon']).copy()
    print(f'Empreendimentos com coords válidas: {len(emp):,}')

    dists, idxs = tree_inmet.query(emp[['lat', 'lon']].to_numpy(), k=1)
    emp['inmet_dist_km'] = dists * 111.32  # graus → km (aprox. Brasil)

    emp = emp.reset_index(drop=True)
    CLIMA_COLS = [
        'CODIGO (WMO)', 'ESTACAO', 'UF',
        'temp_media', 'temp_std', 'temp_max', 'temp_min',
        'umid_media', 'prec_total', 'prec_max_h',
        'radiacao_med', 'vento_med', 'rajada_max',
    ]
    clima_match = silver_inmet.iloc[idxs][CLIMA_COLS].reset_index(drop=True)
    clima_match.columns = [
        'inmet_codigo' if c == 'CODIGO (WMO)' else f'inmet_{c}'
        for c in CLIMA_COLS
    ]
    emp = pd.concat([emp, clima_match], axis=1)
    print(f'✅ Join clima OK. Distância média: {emp["inmet_dist_km"].mean():.1f} km')

    # ── 2) Saturação: datacenters a ≤ 100 km ──────────────────────────────────
    RAIO_KM = 100
    dc_valid = silver_dc.dropna(subset=['LATITUDE', 'LONGITUDE'])
    dc_xy    = dc_valid[['LATITUDE', 'LONGITUDE']].to_numpy()

    if dc_xy.size == 0:
        print('⚠️  Nenhum DC com coords válidas — dc_proximos = 0.')
        emp['dc_proximos'] = 0
    else:
        tree_dc     = cKDTree(dc_xy)
        raio_graus  = RAIO_KM / 111.32
        emp_has_coords = emp.dropna(subset=['lat', 'lon'])
        n_list = tree_dc.query_ball_point(
            emp_has_coords[['lat', 'lon']].to_numpy(), r=raio_graus
        )
        temp = pd.Series(
            [len(x) for x in n_list], index=emp_has_coords.index
        )
        emp['dc_proximos'] = temp.reindex(emp.index, fill_value=0)
    print(f'✅ Saturação OK. Empreendimentos com DC<{RAIO_KM}km: '
          f'{(emp["dc_proximos"] > 0).sum():,}')

    # ── 3) Cobertura ANATEL por UF ────────────────────────────────────────────
    emp = emp.merge(
        silver_anatel[['UF', 'anatel_total']],
        left_on='SigUFPrincipal', right_on='UF', how='left'
    ).drop(columns=['UF'])
    emp['anatel_total'] = emp['anatel_total'].fillna(0)
    print(f'✅ ANATEL OK. Mediana contratos/UF: {emp["anatel_total"].median():.0f}')

print(f'Dataset enriquecido: {emp.shape}')
emp.head(3)

## 2.6 — Gold: feature store por empreendimento

Persiste `gold_empreendimentos_features` — tabela de entrada para a Etapa 3 (ML).

In [ ]:
if not SKIP_RECOMPUTE:
    FEATURES_PERSIST = [
        'CodCEG', 'NomEmpreendimento', 'SigUFPrincipal', 'SigTipoGeracao',
        'DscOrigemCombustivel', 'DscFaseUsina', 'DscTipoOutorga',
        'MdaPotenciaOutorgadaKw', 'MdaPotenciaFiscalizadaKw', 'MdaGarantiaFisicaKw',
        'DatEntradaOperacao', 'lat', 'lon', 'eh_renovavel',
        'inmet_codigo', 'inmet_dist_km',
        'inmet_temp_media', 'inmet_temp_std', 'inmet_temp_max', 'inmet_temp_min',
        'inmet_umid_media', 'inmet_prec_total', 'inmet_prec_max_h',
        'inmet_radiacao_med', 'inmet_vento_med', 'inmet_rajada_max',
        'dc_proximos', 'anatel_total',
    ]
    cols_ok = [c for c in FEATURES_PERSIST if c in emp.columns]
    gold_features = emp[cols_ok].copy()

    if 'DatEntradaOperacao' in gold_features.columns:
        gold_features['DatEntradaOperacao'] = gold_features['DatEntradaOperacao'].apply(
            lambda x: x.isoformat() if pd.notna(x) else None
        )

    col_gold = db['gold_empreendimentos_features']
    col_gold.drop()
    col_gold.insert_many(gold_features.to_dict(orient='records'))
    col_gold.create_index([('SigUFPrincipal', 1), ('DscFaseUsina', 1)])
    print(f'✅ gold_empreendimentos_features: {col_gold.estimated_document_count():,} docs')

emp.head(3)

---
## 2.7 — Análise Exploratória de Dados (EDA)

Visualizações sobre as quatro fontes integradas.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

sns.set_theme(style='whitegrid', palette='viridis')
plt.rcParams['figure.figsize'] = (12, 5)

# ── Resumo estatístico ────────────────────────────────────────────────────────
print('=== Resumo numérico (kW) ===')
print(df_siga[['MdaPotenciaOutorgadaKw', 'MdaPotenciaFiscalizadaKw', 'MdaGarantiaFisicaKw']]
      .describe().round(2))

print('\n=== Nulos por coluna (%) — Top 10 ===')
print((df_siga.isna().mean() * 100).round(2).sort_values(ascending=False).head(10))

In [ ]:
# ── Distribuição por tipo de geração ─────────────────────────────────────────
tipo_count = df_siga['SigTipoGeracao'].value_counts()
ax = tipo_count.plot(kind='bar',
                     color=sns.color_palette('viridis', len(tipo_count)))
ax.set_title('Quantidade de empreendimentos por tipo de geração — Brasil')
ax.set_xlabel('Tipo de geração')
ax.set_ylabel('Qtd. empreendimentos')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}',
                (p.get_x() + p.get_width() / 2, p.get_height()),
                ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ── Top 15 UFs por capacidade outorgada (MW) ─────────────────────────────────
cap_uf = (df_siga.groupby('SigUFPrincipal')['MdaPotenciaOutorgadaKw']
          .sum().div(1000).sort_values(ascending=False).head(15))
ax = cap_uf.plot(kind='barh',
                 color=sns.color_palette('mako', len(cap_uf)))
ax.invert_yaxis()
ax.set_title('Top 15 UFs por capacidade outorgada (MW)')
ax.set_xlabel('MW')
ax.set_ylabel('UF')
plt.tight_layout()
plt.show()

In [ ]:
# ── Distribuição percentual por fase ──────────────────────────────────────────
fase = df_siga['DscFaseUsina'].value_counts(normalize=True).mul(100).round(2)
ax = fase.plot(kind='bar',
               color=['#2ca02c', '#ff7f0e', '#d62728', '#7f7f7f'])
ax.set_title('Distribuição percentual por fase do empreendimento')
ax.set_ylabel('%')
ax.set_xlabel('')
for p in ax.patches:
    ax.annotate(f'{p.get_height():.1f}%',
                (p.get_x() + p.get_width() / 2, p.get_height()),
                ha='center', va='bottom')
plt.tight_layout()
plt.show()

In [ ]:
# ── Mapa interativo: empreendimentos em operação ──────────────────────────────
df_mapa = emp.dropna(subset=['lat', 'lon'])
df_mapa = df_mapa[df_mapa['DscFaseUsina'] == 'Operação'].sample(
    min(3000, len(df_mapa)), random_state=42
)
fig = px.scatter_mapbox(
    df_mapa, lat='lat', lon='lon',
    color='SigTipoGeracao',
    size='MdaPotenciaOutorgadaKw',
    hover_name='NomEmpreendimento',
    hover_data={'SigUFPrincipal': True,
                'MdaPotenciaOutorgadaKw': ':.0f',
                'dc_proximos': True},
    zoom=3, height=600,
    title='Empreendimentos em Operação (amostra) — cor: tipo de geração'
)
fig.update_layout(mapbox_style='open-street-map',
                  margin=dict(r=0, t=30, l=0, b=0))
fig.show()

In [ ]:
# ── Heatmap: perfil climático por UF ─────────────────────────────────────────
UF_LIST = [
    'AC','AL','AM','AP','BA','CE','DF','ES','GO','MA','MG','MS','MT',
    'PA','PB','PE','PI','PR','RJ','RN','RO','RR','RS','SC','SE','SP','TO'
]
clima_uf = silver_inmet.groupby('UF')[
    ['temp_media', 'umid_media', 'prec_total', 'radiacao_med', 'vento_med']
].mean()
clima_uf = clima_uf.reindex(UF_LIST).dropna()
fig, ax = plt.subplots(figsize=(11, 8))
sns.heatmap(clima_uf.div(clima_uf.max()),
            annot=clima_uf.round(1), fmt='.1f',
            cmap='RdYlGn_r',
            cbar_kws={'label': 'normalizado (0-1)'},
            ax=ax)
ax.set_title('Perfil climático médio por UF (fonte INMET)')
plt.tight_layout()
plt.show()

In [ ]:
# ── Cobertura ANATEL × Capacidade ANEEL por UF ───────────────────────────────
fig_uf = (
    df_siga.groupby('SigUFPrincipal')['MdaPotenciaOutorgadaKw']
    .sum().div(1000)
    .reset_index(name='cap_MW')
    .merge(silver_anatel[['UF', 'anatel_total']],
           left_on='SigUFPrincipal', right_on='UF')
)
fig = px.scatter(
    fig_uf, x='anatel_total', y='cap_MW', text='UF',
    size='cap_MW', color='cap_MW', color_continuous_scale='viridis',
    title='Cobertura telecom (ANATEL) × Capacidade energética (ANEEL) por UF',
    labels={
        'anatel_total': 'Contratos ANATEL (proxy de fibra)',
        'cap_MW': 'Capacidade outorgada (MW)',
    }
)
fig.update_traces(textposition='top center')
fig.show()

---

## ✅ Etapa 2 — Concluída

| Coleção Silver/Gold                  | Conteúdo                                   |
|--------------------------------------|--------------------------------------------|
| `silver_siga`                        | ANEEL limpo, tipado, coordenadas WGS-84    |
| `silver_inmet`                       | Estatísticas climáticas por estação        |
| `silver_anatel_uf`                   | Contratos ANATEL por UF                    |
| `silver_datacenters`                 | DCs com geocodificação por cidade          |
| `gold_empreendimentos_features`      | Feature store completa para ML (Etapa 3)   |

> **Próximo passo:** execute a Etapa 3 (`Etapa3_ML_Modelos.ipynb`) que carrega  
> `gold_empreendimentos_features` e treina os modelos de classificação e regressão.